In [ ]:
import sys
sys.path.append("..")

import pickle
import time
from itertools import product
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import shap
from mlresearch.datasets import BinaryDatasets, ContinuousCategoricalDatasets
from tabpfn import TabPFNRegressor
from explainerpfn.base import ExplainerPFN
from explainerpfn.train import SyntheticDataGenerator
from explainerpfn.train._activations import identity

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Total and free memory in bytes
    free, total = torch.cuda.mem_get_info()
    print(f"Total memory: {total / 1024**2:.2f}MB")
    print(f"Free memory: {free / 1024**2:.2f}MB")
else:
    print("No GPU available. Training will run on CPU.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random_state = 42
max_samples = 20000
max_features = 15
model_path = "../../explainerpfn_model_NOV_15.ckp"

# Collect datasets

In [ ]:
datasets = BinaryDatasets().download()
datasets.content_ += ContinuousCategoricalDatasets().download().content_
content_processed = []
for name, df in datasets:
    df.columns = df.columns.astype(str)
    df = df.loc[:, ~df.columns.str.startswith("cat")].copy()
    if df.target.nunique() > 2:
        df.target = (df.target == df.target.mode().iloc[0]).astype(int)
    
    if df.shape[1] < max_features and df.shape[1] > 2 and df.shape[0] < max_samples:
        content_processed.append((name, df))

datasets.content_ = content_processed
datasets.summarize_datasets()

# Collect explanations

In [ ]:
def few_shot_explanations(X_train, y_train, explanations_train, X_test, y_test, model, n_samples=5, random_state=42):

    X_train_sample, _, y_train_sample, _, explanations_sample, _ = train_test_split(
        X_train, y_train, explanations_train,
        train_size=n_samples, 
        random_state=random_state
    )
    Xy_sample = np.concatenate([X_train_sample, y_train_sample.reshape(-1, 1)], axis=1)
    Xy_test = np.concatenate([X_test, y_test.reshape(-1, 1)], axis=1) 

    tabpfn_preds = np.zeros(X_test.shape, dtype=float)
    for i in range(X_test.shape[1] - 1): 
        exp_target = explanations_sample[:, i]

        model.fit(Xy_sample, exp_target)

        pred = model.predict(Xy_test)
        tabpfn_preds[:, i] = pred

    return tabpfn_preds


In [ ]:
def run_experiments_across_datasets(datasets, model_path, base_model, few_shot_sample_size, few_shot_predictors, random_state=42):
    results = {}

    for dataset_name, df in tqdm(datasets):

        results[dataset_name] = {}
        X = df.drop(columns=["target"]).values
        y = df["target"].values

        # Train a model and get predictions
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)
        model = clone(base_model)
        model.fit(X_train, y_train)
        y_train_scores = model.predict_proba(X_train)[:, 1]
        y_test_scores = model.predict_proba(X_test)[:, 1]
        results[dataset_name]["X_train"] = X_train
        results[dataset_name]["X_test"] = X_test
        results[dataset_name]["y_train"] = y_train
        results[dataset_name]["y_test"] = y_test
        results[dataset_name]["model"] = model
        results[dataset_name]["y_train_scores"] = y_train_scores
        results[dataset_name]["y_test_scores"] = y_test_scores

        shap_explanations = shap.Explainer(model.named_steps["mlp"].predict_proba, model.named_steps["scaler"].transform(X_train))
        shap_values_train = shap_explanations(model.named_steps["scaler"].transform(X_train)).values[:, :, -1]
        shap_values_test = shap_explanations(model.named_steps["scaler"].transform(X_test)).values[:, :, -1]
        results[dataset_name]["shap"] = shap_values_test

        start_time = time.time()
        xai = ExplainerPFN(model_path=model_path, random_state=random_state, device=device)
        xai.fit(np.concatenate([X_train, X_test], axis=0), np.concatenate([y_train_scores, y_test_scores], axis=0))
        exp_pfn_values = xai.predict(X_test, y_test_scores)
        end_time = time.time()
        results[dataset_name]["explainerpfn_time"] = end_time - start_time
        results[dataset_name]["explainerpfn"] = exp_pfn_values / np.sqrt(X.shape[1])  # Normalize by number of features
        results[dataset_name]["explainerpfn_statistical+additive"] = xai.apply_correction(y_test_scores, exp_pfn_values, kind=["statistical", "additive"])
        results[dataset_name]["explainerpfn_statistical+multiplicative"] = xai.apply_correction(y_test_scores, exp_pfn_values, kind=["statistical", "multiplicative"])

        for n_samples, (model_name, model) in product(few_shot_sample_size, few_shot_predictors.items()):
            start_time = time.time()
            model = clone(model)
            exp_few_shot = few_shot_explanations(X_train, y_train, shap_values_train, X_test, y_test, model, n_samples=n_samples, random_state=random_state)
            end_time = time.time()
            results[dataset_name][f"{model_name}_n{n_samples}_time"] = end_time - start_time
            results[dataset_name][f"{model_name}_n{n_samples}"] = exp_few_shot
        
    return results

In [ ]:
# Structured as Dict[dataset_name, Dict[model_name, explanations]]
base_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(hidden_layer_sizes=(12, 12), max_iter=5000, random_state=random_state))
    ]
)
few_shot_sample_size = [2, 4, 6, 8, 10]
few_shot_predictors = {
    "tabpfn": TabPFNRegressor(model_path="tabpfn-v2-regressor.ckpt", random_state=random_state, device=device),
    "mlp": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(hidden_layer_sizes=(12, 12), max_iter=5000, random_state=random_state))
        ]
    ),
    "rfr": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("rfr", RandomForestRegressor(n_estimators=500, random_state=random_state))
        ]
    )
}


# Get results across datasets
results = run_experiments_across_datasets(
    datasets=datasets,
    model_path=model_path,
    base_model=base_model,
    few_shot_sample_size=few_shot_sample_size,
    few_shot_predictors=few_shot_predictors,
    random_state=random_state
)

# Save results
with open("results_across_datasets_mlp.pkl", "wb") as f:
    pickle.dump(results, f)

# Getting new results

This section was set up to analyse new versions of ExplainerPFN. Feel free to ignore.

In [ ]:
with open("results_across_datasets.pkl", "rb") as f:
    results = pickle.load(f)


In [ ]:
model_path = "../../explainerpfn_model_NOV_15.ckp"

# Compute results for new explainerpfn model individually
for dataset_name, df in tqdm(datasets):
    X_train = results[dataset_name]["X_train"]
    X_test = results[dataset_name]["X_test"]
    y_train_scores = results[dataset_name]["y_train_scores"]
    y_test_scores = results[dataset_name]["y_test_scores"]

    model_name = model_path.split("/")[-1].replace(".ckp", "")
    xai = ExplainerPFN(model_path=model_path, random_state=random_state)
    xai.fit(np.concatenate([X_train, X_test], axis=0), np.concatenate([y_train_scores, y_test_scores], axis=0))
    exp_pfn_values = xai.predict(X_test, y_test_scores)
    results[dataset_name][model_name] = exp_pfn_values / np.sqrt(X_test.shape[1])  # Normalize by number of features
    
    results[dataset_name][f"{model_name}_statistical+additive"] = xai.apply_correction(y_test_scores, exp_pfn_values, kind=["statistical", "additive"])

# Results Analysis

In [ ]:
def top_k_jaccard_similarity(arr1, arr2, k=3):
    if arr1.ndim == 1:
        arr1 = arr1.reshape(1, -1)
    if arr2.ndim == 1:
        arr2 = arr2.reshape(1, -1)

    feats1 = np.argpartition(arr1, -k, axis=1)[:, -k:]
    feats2 = np.argpartition(arr2, -k, axis=1)[:, -k:]

    intersection = [np.intersect1d(row1, row2).shape[0] for row1, row2 in zip(feats1, feats2)]
    union = [row1.shape[0] + row2.shape[0] - intersect for row1, row2, intersect in zip(feats1, feats2, intersection)]
    return np.array(intersection) / np.array(union)


def get_results_tables(
    results_dict,
    jaccard_perc=1/3
):
    model_names = [
        key for key in list(results_dict.values())[0]
        if not (key.startswith("X_") or key.startswith("y_") or key == "model")
    ]

    # Compute metrics
    mse = {}
    corr = {}
    jaccard = {}
    for dataset_name in results_dict:
        mse[dataset_name] = {}
        corr[dataset_name] = {}
        jaccard[dataset_name] = {}
        jaccard_k = max(1, int(results_dict[dataset_name]["X_test"].shape[1] * jaccard_perc))
        shap_values_test = results_dict[dataset_name]["shap"]
        for model_name in model_names:
            exp_values = results_dict[dataset_name][model_name]
            mse[dataset_name][model_name] = np.mean((exp_values - shap_values_test) ** 2)
            corr[dataset_name][model_name] = np.corrcoef(exp_values.ravel(), shap_values_test.ravel())[0, 1]
            jaccard[dataset_name][model_name] = np.mean(top_k_jaccard_similarity(exp_values, shap_values_test, k=jaccard_k))

    dataframes = []
    for metric in [mse, corr, jaccard]:
        df_metric = pd.DataFrame(metric).drop(["explainerpfn_statistical+multiplicative", "explainerpfn"])
        df_metric["Method"] = df_metric.index.map(lambda x: "_".join(x.split("_")[:-1]))
        df_metric["Samples"] = df_metric.index.map(lambda x: x.split("_n")[-1]).map(lambda x: int(x) if x.isdigit() else 0)
        df_metric = df_metric[df_metric["Method"] != "shap"]
        df_metric = df_metric.set_index(["Method", "Samples"]).sort_index()
        dataframes.append(df_metric.copy())
    
    df_mse, df_corr, df_jaccard = dataframes
    return df_mse, df_corr, df_jaccard

In [ ]:
df_mse, df_corr, df_jaccard = get_results_tables(results)

In [ ]:
df_mse

In [ ]:
df_corr

In [ ]:
df_jaccard

In [ ]:
# l2 - distance (comparable to mse)
# on correlation - add more weight to more important features
# columntransformer: feature # explode if transforming categorical to onehot
#       - The correlation between features might come out incorrect
# sensitivity: continuous features are more sensitive.
#     - shapley computations vary according to model uncertainty
#     - Analysis of the ranked order of features: top-k jaccard similarity
#     - Shift distance - top 3 (for example) jaccard similarity
#         - If you consider all features you get noisy measurements
# analyze individually between correct / incorrect predictions (FP, FN) vs (TN, TP)
# analyze individually between certain / uncertain predictions
# Beeswarm plots
# Case study suggestion - Diabetes (not useful), ACS Income (very useful)

# Generalization over different types of classifiers

In [ ]:
# Structured as Dict[dataset_name, Dict[model_name, explanations]]
base_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("rfc", RandomForestClassifier(n_estimators=500, random_state=random_state))
    ]
)
few_shot_sample_size = [2, 4, 6, 8, 10]
few_shot_predictors = {
    "tabpfn": TabPFNRegressor(model_path="tabpfn-v2-regressor.ckpt", random_state=random_state, device=device),
    "mlp": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(hidden_layer_sizes=(12, 12), max_iter=5000, random_state=random_state))
        ]
    ),
    "rfr": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("rfr", RandomForestRegressor(n_estimators=500, random_state=random_state))
        ]
    )
}


# Get results across datasets
results = run_experiments_across_datasets(
    datasets=datasets,
    model_path=model_path,
    base_model=base_model,
    few_shot_sample_size=few_shot_sample_size,
    few_shot_predictors=few_shot_predictors,
    random_state=random_state
)

# Save results
with open("results_across_datasets_random_forest.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
df_mse, df_corr = get_results_tables(results)

In [ ]:
df_mse

In [ ]:
df_corr

# DAG Reconstruction

In [ ]:
def get_all_explanations(xai_model, df):
    explanations = {}
    for target_col in tqdm(df.columns): 
        X = df.drop(columns=[target_col]).values
        y = df[target_col].values

        xai = clone(xai_model)
        xai.fit(X, y)
        exp_values = xai.predict(X, y)
        exp_values = xai.apply_correction(y, exp_values, kind=["statistical", "additive"])
        explanations[target_col] = pd.DataFrame(exp_values, columns=df.drop(columns=[target_col]).columns)
    return explanations

In [ ]:
def make_dag_from_explanations(explanations, threshold=75):
    G = nx.DiGraph()
    edges = []
    for target in explanations:
        exp_df = explanations[target].abs().mean()
        edges.extend([{"source": source, "target": target, "weight": weight} for source, weight in exp_df.items()])

    # Filter edges by threshold
    all_weights = [edge["weight"] for edge in edges]
    threshold = np.percentile(all_weights, threshold)
    edges = [edge for edge in edges if edge["weight"] > threshold]
    for edge in edges:
        G.add_edge(edge["source"], edge["target"], weight=edge["weight"])
    return G

In [ ]:
def plot_dag(G, colors=None):
    colors = colors if colors is not None else "#A0CBE2"
    plt.figure(figsize=(4, 4))
    pos = nx.shell_layout(G)
    nx.draw(
        G, pos, with_labels=True, arrows=True, node_size=700, node_color=colors
    )
    edge_labels = nx.get_edge_attributes(G, "weight")
    formatted_edge_labels = {k: f"{v:.2f}" for k, v in edge_labels.items()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=formatted_edge_labels)
    plt.show()

In [ ]:

generator = SyntheticDataGenerator(
    n_dags=[4, 4],
    n_nodes=[2, 3],
    n_features=[6, 7],
    successor_as_target=[True], 
    random_state=random_state
)
params = generator._sample_params()
df, dag = generator._dataset(**params, activations=[identity])
df.drop(columns=df.columns[-1], inplace=True)  # Drop target column
df.columns = df.columns.map(lambda x: x.split("_")[1])
df = df.T.sort_index().T
df

In [ ]:
all_nodes = list(dag.nodes)
nodes_in_data = [int(col) for col in df.columns]
nodes_not_in_data = [node for node in all_nodes if node not in nodes_in_data]
colors = {
    **{node: "#CA9E98" for node in nodes_not_in_data},
    **{node: "#A0CBE2" for node in nodes_in_data}
}
colors = [colors[node] for node in dag.nodes]

In [ ]:
plot_dag(dag, colors=colors)

In [ ]:
xai_model = ExplainerPFN(model_path=model_path, random_state=random_state, device=device)
explanations = get_all_explanations(xai_model, df)

In [ ]:
# Plot explanation graph
G_exp = make_dag_from_explanations(explanations, threshold=75)
plot_dag(G_exp)

In [ ]:
nx.graph_edit_distance(
    dag.to_undirected(), 
    make_dag_from_explanations(explanations, threshold=75).to_undirected()
)

In [ ]:
thresholds = list(range(0, 101, 5))
edit_distances = [
    nx.graph_edit_distance(
        dag.to_undirected(), 
        make_dag_from_explanations(explanations, threshold=t).to_undirected()
    )
    for t in thresholds
]
plt.plot(thresholds, edit_distances)

In [ ]:
# Generate several DAGs and get explanation graphs
xai_model = ExplainerPFN(model_path=model_path, random_state=random_state, device=device)
dags = []
dfs = []
explanations = []
for _ in tqdm(range(50)):
    params = generator._sample_params()
    df, dag = generator._dataset(**params, activations=[identity])
    df.drop(columns=df.columns[-1], inplace=True)  # Drop target column
    df.columns = df.columns.map(lambda x: x.split("_")[1])
    df = df.T.sort_index().T
    dfs.append(df)
    dags.append(dag)
    explanations.append(get_all_explanations(xai_model, df))

with open("synthetic_dags_explanations.pkl", "wb") as f:
    pickle.dump((dags, dfs, explanations), f)

In [ ]:
# dags, dfs, explanations = pickle.load(open("synthetic_dags_explanations.pkl", "rb"))

In [ ]:
thresholds = np.arange(0, 100, 5)
all_edit_distances = []
for dag, df, explanations in tqdm(zip(dags, dfs, explanations)):
    edit_distances = [
        nx.graph_edit_distance(
            dag.to_undirected(), 
            make_dag_from_explanations(explanations, threshold=t).to_undirected()
        )
        for t in thresholds
    ]
    all_edit_distances.append(edit_distances)


# Case Study: ACS Income

In [ ]:
# Tasks:
# 1. Recover causal structure
# 2. Help determine which ML model are more appropriate for a given dataset, based on how well they align with Shapley value expectations
# 3. Are faithful to regular feature importance measures
# 4. Compare different observations explanations
